# Posterior predictive checks and residual diagnostics

After a fit: does the model reproduce the observed outcome's summary statistics, and do the
residuals look like the noise the likelihood assumed? `posterior_predictive` simulates replicate
outcomes from posterior draws and compares each `Statistic` of the data against its predictive
interval and tail probability; `residuals` runs Durbin–Watson, Ljung–Box, normality, and
Breusch–Pagan tests on the posterior-mean residuals, each with its statistic, p-value, and the
`N` it used.

In [ ]:
import numpy as np

from axiom.core import Prior, Unsupported
from axiom.diagnose import (
    DEFAULT_STATISTICS, PPCResult, ResidualReport, ResidualTest, Statistic, StatisticCheck,
    UnitResiduals, posterior_predictive, residuals,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import HillKernel, fit

AMP = Prior(family="lognormal", hyper={"mu": 0.0, "sigma": 0.5})
world = surface_world(n_units=4, n_periods=12, treatments=("a",), kernels=HillKernel(reference_dose=1.0, amplitude_prior=AMP), intercept="shared", noise_sd=0.3, seed=7, doses=DosePlan(zero_fraction=0.2))
res = fit(world.spec, world.panel, backend="laplace", draws=100, chains=1, seed=0)
print("converged:", res.converged, "| draws:", res.n_draws())

## `posterior_predictive`

`DEFAULT_STATISTICS` maps names to `Statistic` callables (mean, sd, min, max, lag-1
autocorrelation, ...). Each `StatisticCheck` carries the observed value, the predictive `eti`
interval with its mass, the one- and two-sided tail probabilities over `n` replicates, and
`extreme` at the stated alpha. A statistic that returns a non-finite value is listed in
`skipped` rather than silently dropped.

In [ ]:
print("default statistics:", list(DEFAULT_STATISTICS))
ppc = posterior_predictive(res, n_draws=80, seed=1)
assert isinstance(ppc, PPCResult)
for s in ppc.statistics:
    assert isinstance(s, StatisticCheck)
    print(f"  {s.name:8s} observed={s.observed:8.3f} interval=[{s.interval.lower:8.3f}, {s.interval.upper:8.3f}] p2={s.p_two_sided:.3f} extreme={s.extreme}")
print("extreme:", ppc.extreme_statistics, "| round-trips:", PPCResult.from_json(ppc.to_json()) == ppc)


def spread(y: np.ndarray) -> float:
    return float(np.max(y) - np.min(y))


custom: dict[str, Statistic] = {"spread": spread, "nan": lambda y: float("nan")}
ppc2 = posterior_predictive(res, statistics=custom, n_draws=40, seed=2)
assert isinstance(ppc2, PPCResult)
print("custom:", [s.name for s in ppc2.statistics], "| skipped:", ppc2.skipped)

## `residuals`

Per-unit `UnitResiduals` (mean and sd over the unit's periods) and a `ResidualTest` per check.
Durbin–Watson has no p-value (its bounds depend on the design) and is reported as a statistic
only; Ljung–Box runs at each requested lag below the number of periods, and the rest are
skipped with a reason. `flagged` lists the tests with `p < alpha`.

In [ ]:
rr = residuals(res, lags=(1, 4, 20), alpha=0.05)
assert isinstance(rr, ResidualReport)
print(f"n={rr.n} ({rr.n_units} units x {rr.n_periods} periods), residual sd={rr.residual_sd:.3f}")
for u in rr.units:
    assert isinstance(u, UnitResiduals)
    print(f"  {u.unit}: mean={u.mean:7.3f} sd={u.sd:.3f} n={u.n}")
for t in rr.tests:
    assert isinstance(t, ResidualTest)
    p = "n/a" if t.p_value is None else f"{t.p_value:.3f}"
    print(f"  {t.name:14s} stat={t.statistic:8.3f} p={p} n={t.n} {t.note}")
print("skipped:", rr.skipped, "| flagged:", rr.flagged)
print("round-trips:", ResidualReport.from_json(rr.to_json()) == rr)
declined = fit(world.spec, world.panel, backend="no-such-backend")
print("no posterior ->", type(residuals(declined)).__name__, "/", type(posterior_predictive(declined)).__name__)
assert isinstance(residuals(declined), Unsupported)